# Financial Transaction Risk Scoring Engine - Business Analysis & Prioritization

This notebook provides a business-centric interpretation of the outputs from our Financial Transaction Risk & Anomaly Engine. The objective is to translate raw machine learning outputs (anomaly probabilities) into actionable business categories, helping fraud operations and business teams prioritize high-risk cases effectively.

## 1. Load Risk Analysis Results
First, we load the generated risk scoring predictions and summary metrics from the reports directory.

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display, Image

# Ensure correct working directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Load Risk Summary Report
summary_path = "reports/risk_summary.json"
with open(summary_path, 'r') as f:
    summary = json.load(f)

print("=== CONFIGURATION METRICS ===")
print(f"Low Risk Threshold (Probability):  {summary.get('configured_low_risk_threshold')}")
print(f"High Risk Threshold (Probability): {summary.get('configured_high_risk_threshold')}")
print("\n=== TRANSACTION CASE SUMMARY ===")
print(f"Total Test Predictions Evaluated:  {summary.get('total_predictions')}")
print(f"High Risk Cases (Flagged):         {summary.get('high_risk_cases')} ({summary.get('percentage_high_risk')}%)")
print(f"Medium Risk Cases (Review):        {summary.get('medium_risk_cases')} ({summary.get('percentage_medium_risk')}%)")
print(f"Low Risk Cases (Approved):         {summary.get('low_risk_cases')} ({summary.get('percentage_low_risk')}%)")

## 2. Risk Scoring Visualizations
We render the generated visualizations to understand the risk profile and distribution of transactions.

In [ ]:
print("=== 2.1 Risk Level Distribution (Bar Chart) ===")
display(Image(filename='reports/figures/risk_distribution.png'))

In [ ]:
print("=== 2.2 Risk Score Histogram (0-100 Scale) ===")
display(Image(filename='reports/figures/risk_score_histogram.png'))

In [ ]:
print("=== 2.3 Model Prediction Probability Density ===")
display(Image(filename='reports/figures/probability_distribution.png'))

## 3. Business Prioritization Framework & Actionable Strategies

Our Risk Scoring Engine categorizes every incoming transaction into one of three risk tiers based on model probability thresholds. This framework allows business and fraud operations teams to design specific response playbooks for each category.

### 3.1. High Risk Tier (Risk Score $\ge$ 60)
- **Summary Status**: Highly anomalous transactions exhibiting signatures matching known threat profiles (e.g., extreme amount ratio compared to history, suspicious international geography, or large transfers at odd night hours).
- **Business Impact**: High probability of fraud/anomaly (1.90% alert rate in test split).
- **Prioritization Playbook**:
  1. **Real-time Auto-Decline**: These transactions should be automatically blocked at the gateway level to prevent financial loss. The customer is notified via SMS/Push notification immediately.
  2. **Card Restructuring**: Automatically freeze the card or account used for the transaction, requiring multi-factor authentication (MFA) or identity verification via the banking app to unblock.
  3. **High-Priority Review**: If auto-decline is not feasible (e.g., due to VIP user status or specific transaction rules), direct these cases to the top of the manual review queue with a target SLA of **under 15 minutes**.

### 3.2. Medium Risk Tier (20 $\le$ Risk Score < 60)
- **Summary Status**: Borderline transactions showing minor deviations from average user behavior (e.g., moderate amount spikes or transaction channels not typical but not flagrantly suspicious).
- **Business Impact**: In the current test set, our Random Forest classifier is highly confident (producing 0% Medium Risk cases). However, during model drifts or updates, this tier serves as a safety net to prevent false positives and negative customer friction.
- **Prioritization Playbook**:
  1. **Soft-Block with SMS MFA**: Rather than declining the transaction, trigger an interactive challenge (SMS passcode or biometrics confirmation) to verify the user is authorizing the transaction in real-time.
  2. **Standard Review Queue**: Place in a manual review queue for fraud analysts to investigate. Analysts can run secondary checks (e.g. IP lookup, device fingerprint inspection, cross-referencing merchant history) within a **4-hour SLA**.

### 3.3. Low Risk Tier (Risk Score < 20)
- **Summary Status**: Standard transaction behaviors matching the baseline profile of the customer.
- **Business Impact**: Represents **98.10%** of all transactions, establishing that the vast majority of customer payments pass without any friction.
- **Prioritization Playbook**:
  1. **Straight-Through Processing (STP)**: Approved instantly at the payment gateway with no extra friction.
  2. **Passive Logging**: Store telemetry data for future rolling average calculations.

## 4. Threshold Tuning & Policy Optimization

Thresholds are configurable in `config/risk_scoring_config.json` to allow business teams to balance **fraud detection rate** vs. **operational queue capacity**:

- **Tightening Policy (Ops constrained)**: If the fraud operations team is overwhelmed by review alerts, raise the `high_risk_threshold` (e.g., to `0.80`) and the `low_risk_threshold` (e.g., to `0.30`). This narrows the volume of generated alerts, filtering for only the most severe risk signatures.
- **Loosening Policy (Loss prevention priority)**: If fraud losses are spiking and the business is willing to tolerate higher customer friction/analyst reviews, lower the `high_risk_threshold` (e.g., to `0.50`). This expands the net to intercept more suspicious activities, improving fraud recall.